# YeastCaduceus — Phase 1: Data Preprocessing

**Purpose:** Convert raw `.softmask` FASTAs and BigWig tracks into HuggingFace Datasets ready for:
- **Phase 2 (MLM):** 8,192bp sliding windows from 165_Saccharomycetales corpus
- **Phase 3 (Supervised):** Paired (sequence, coverage) tensors from R64 + BigWig tracks

**Key design choices vs Shorkie:**
| Parameter | Shorkie | YeastCaduceus |
|---|---|---|
| Window size | 16,384bp | 8,192bp (CAD2-Small hard limit) |
| Stride | 4,096bp (25%) | 2,048bp (25%) |
| Repeat filter | >7% lowercase | >7% lowercase (same) |
| Coverage bins | 896 × 16bp | 512 × 16bp |
| Chrom split | same | same |

**Chromosome split (R64, matching Shorkie exactly):**
- `val`: chrXI, chrXIII, chrXV
- `test`: chrXII, chrXIV, chrXVI
- `train`: all remaining R64 chroms + all non-R64 genomes

**Run order:** Mount Drive first (Cell 1), then run cells top to bottom.

---

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — Setup: mount Drive, install deps, define all paths and constants
# ─────────────────────────────────────────────────────────────────────────────

from google.colab import drive
drive.mount('/content/drive')

!pip install datasets pyBigWig pyfaidx biopython tqdm -q

import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE          = Path('/content/drive/MyDrive/shorkie/data/backup')
OUT_BASE      = Path('/content/drive/MyDrive/yeastcaduceus/data')

FASTA_R64     = BASE / 'genomes/R64'
FASTA_165     = BASE / 'genomes/165_Saccharomycetales'
FASTA_80      = BASE / 'genomes/80_strains'
BIGWIG_DIR    = BASE / 'supervised/bigwigs'

OUT_MLM       = OUT_BASE / 'mlm_dataset'        # HF Dataset for Phase 2
OUT_SUP_SEQ   = OUT_BASE / 'supervised_seqs'    # FASTA windows for Phase 3
OUT_SUP_COV   = OUT_BASE / 'supervised_cov'     # BigWig coverage arrays
OUT_MANIFEST  = OUT_BASE / 'manifests'          # CSV manifests for tracking

for d in [OUT_MLM, OUT_SUP_SEQ, OUT_SUP_COV, OUT_MANIFEST]:
    d.mkdir(parents=True, exist_ok=True)

# ── Constants ─────────────────────────────────────────────────────────────────
WINDOW_SIZE   = 8192    # CAD2-Small hard limit
STRIDE        = 2048    # 25% stride — same ratio as Shorkie (4096/16384)
REPEAT_THRESH = 0.07    # drop windows with >7% lowercase (repetitive) — Shorkie threshold
N_COV_BINS    = WINDOW_SIZE // 16   # 512 bins × 16bp = 8192bp (Shorkie: 896 × 16bp = 14336bp)
MLM_MASK_PROB = 0.15    # BERT-style, same as Shorkie and PlantCAD2

# Chromosome split — matches Shorkie exactly for fair benchmarking
VAL_CHROMS  = {'chrXI', 'chrXIII', 'chrXV'}
TEST_CHROMS = {'chrXII', 'chrXIV', 'chrXVI'}

# R64 FASTA filename (confirmed in Phase 0)
R64_FASTA = next(FASTA_R64.glob('*.softmask'), None)

print('Paths:')
print(f'  R64 FASTA     : {R64_FASTA}')
print(f'  165 genomes   : {FASTA_165}')
print(f'  BigWig dir    : {BIGWIG_DIR}')
print(f'  Output base   : {OUT_BASE}')
print()
print('Constants:')
print(f'  Window        : {WINDOW_SIZE}bp')
print(f'  Stride        : {STRIDE}bp ({int(100*STRIDE/WINDOW_SIZE)}% of window)')
print(f'  Repeat thresh : {REPEAT_THRESH*100}%')
print(f'  Coverage bins : {N_COV_BINS} × 16bp')
print(f'  Val chroms    : {VAL_CHROMS}')
print(f'  Test chroms   : {TEST_CHROMS}')

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.0/187.0 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 59.7 MB/s eta 0:00:00
Paths:
  R64 FASTA     : /content/drive/MyDrive/shorkie/data/backup/genomes/R64/GCA_000146045_2.cleaned.fasta.masked.dust.softmask
  165 genomes   : /content/drive/MyDrive/shorkie/data/backup/genomes/165_Saccharomycetales
  BigWig dir    : /content/drive/MyDrive/shorkie/data/backup/supervised/bigwigs
  Output base   : /content/drive/MyDrive/yeastcaduceus/data

Constants:
  Window        : 8192bp
  Stride        : 2048bp (25% of window)
  Repeat thresh : 7.000000000000001%
  Coverage bins : 512 × 16bp
  Val chroms    : {'chrXIII', 'chrXI', 'chrXV'}
  Test chroms   : {'chrXIV', 'chrXVI', 'chrXII'}


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — MLM Dataset: 165_Saccharomycetales → 8192bp sliding windows
#
# Strategy:
#   - For each .softmask FASTA in 165_Saccharomycetales:
#       - If it's R64 (GCA_000146045): skip chrXI,XIII,XV,XII,XIV,XVI
#         (keep those chromosomes off the training set for fair benchmarking)
#       - All other genomes: all chromosomes go to train
#   - Slide 8192bp windows with 2048bp stride
#   - Drop windows with >7% lowercase (repetitive)
#   - Drop windows shorter than WINDOW_SIZE (chromosome ends)
#   - Record split (train/val/test) — val/test only come from R64
#   - Save as HuggingFace Dataset (arrow format) — efficient for streaming
#
# Output: ~2-4M windows. R64 val: ~20k, test: ~20k. Expect ~10 min on Colab.
# ─────────────────────────────────────────────────────────────────────────────

from datasets import Dataset
import re

def repeat_fraction(seq: str) -> float:
    """Fraction of nucleotides that are lowercase (soft-masked repeats)."""
    alpha = sum(c.isalpha() for c in seq)
    if alpha == 0:
        return 1.0
    lower = sum(c.islower() for c in seq)
    return lower / alpha

def parse_fasta(fasta_path: Path) -> dict:
    """Read a .softmask FASTA. Returns {chrom_name: sequence_string}."""
    chroms = {}
    current_name = None
    current_seq = []
    with open(fasta_path) as f:
        for line in f:
            line = line.rstrip()
            if line.startswith('>'):
                if current_name is not None:
                    chroms[current_name] = ''.join(current_seq)
                # Normalize: take first token, strip leading '>'
                current_name = line[1:].split()[0]
                current_seq = []
            else:
                current_seq.append(line)
    if current_name is not None:
        chroms[current_name] = ''.join(current_seq)
    return chroms

def chrom_to_split(chrom_name: str, is_r64: bool) -> str:
    """Assign split. Val/test only from R64; everything else is train."""
    if not is_r64:
        return 'train'
    # R64 chromosome names may be 'chrXI' or 'XI' — normalise
    c = chrom_name if chrom_name.startswith('chr') else f'chr{chrom_name}'
    if c in VAL_CHROMS:
        return 'val'
    if c in TEST_CHROMS:
        return 'test'
    return 'train'

def windows_from_chrom(chrom_name: str, seq: str, split: str, genome_id: str) -> list:
    """Slide WINDOW_SIZE windows with STRIDE. Filter >7% repeats and short windows."""
    records = []
    for start in range(0, len(seq) - WINDOW_SIZE + 1, STRIDE):
        window = seq[start:start + WINDOW_SIZE]
        if len(window) < WINDOW_SIZE:
            continue
        # Drop N-dominated windows (>50% Ns)
        n_frac = (window.upper().count('N')) / WINDOW_SIZE
        if n_frac > 0.5:
            continue
        # Drop high-repeat windows per Shorkie threshold
        if repeat_fraction(window) > REPEAT_THRESH:
            continue
        records.append({
            'sequence':  window.upper(),  # store uppercase; masking handles repeat info separately
            'genome_id': genome_id,
            'chrom':     chrom_name,
            'start':     start,
            'split':     split,
        })
    return records

In [ ]:


# ── R64 accession prefix for identification ────────────────────────────────────
R64_ACCESSION = 'GCA_000146045'  # confirmed from Phase 0 filename

# ── Main loop ─────────────────────────────────────────────────────────────────
all_records = []
fasta_files = sorted(FASTA_165.glob('*.softmask'))
print(f'Processing {len(fasta_files)} FASTA files...')

stats = {'train': 0, 'val': 0, 'test': 0,
         'dropped_repeat': 0, 'dropped_n': 0, 'dropped_short': 0}

for fasta_path in tqdm(fasta_files, desc='FASTAs'):
    genome_id = fasta_path.stem.split('.')[0]  # e.g. GCA_000146045_2
    is_r64 = R64_ACCESSION in fasta_path.name

    chroms = parse_fasta(fasta_path)

    for chrom_name, seq in chroms.items():
        split = chrom_to_split(chrom_name, is_r64)
        records = windows_from_chrom(chrom_name, seq, split, genome_id)
        all_records.extend(records)
        stats[split] += len(records)

print(f'\nWindow counts after filtering:')
print(f'  train : {stats["train"]:,}')
print(f'  val   : {stats["val"]:,}   (R64 chrXI, chrXIII, chrXV only)')
print(f'  test  : {stats["test"]:,}   (R64 chrXII, chrXIV, chrXVI only)')
print(f'  total : {sum(stats.values()):,}')

# ── Save as HuggingFace Dataset ───────────────────────────────────────────────
print('\nBuilding HuggingFace Dataset...')
for split in ['train', 'val', 'test']:
    split_records = [r for r in all_records if r['split'] == split]
    if not split_records:
        print(f'  ⚠️  {split}: 0 records — skipping')
        continue
    ds = Dataset.from_list(split_records)
    out_path = OUT_MLM / split
    ds.save_to_disk(str(out_path))
    print(f'  ✅ {split}: {len(ds):,} windows → {out_path}')

# ── Save manifest ─────────────────────────────────────────────────────────────
manifest = pd.DataFrame([
    {'split': k, 'n_windows': v} for k, v in stats.items()
])
manifest.to_csv(OUT_MANIFEST / 'mlm_dataset_manifest.csv', index=False)
print(f'\n✅ Cell 2 complete — MLM dataset saved to {OUT_MLM}')

Processing 165 FASTA files...


FASTAs: 100%|██████████| 165/165 [13:25<00:00,  4.88s/it]



Window counts after filtering:
  train : 846,664
  val   : 1,121   (R64 chrXI, chrXIII, chrXV only)
  test  : 1,144   (R64 chrXII, chrXIV, chrXVI only)
  total : 848,929

Building HuggingFace Dataset...


Saving the dataset (0/14 shards):   0%|          | 0/846664 [00:00<?, ? examples/s]

  ✅ train: 846,664 windows → /content/drive/MyDrive/yeastcaduceus/data/mlm_dataset/train


Saving the dataset (0/1 shards):   0%|          | 0/1121 [00:00<?, ? examples/s]

  ✅ val: 1,121 windows → /content/drive/MyDrive/yeastcaduceus/data/mlm_dataset/val


Saving the dataset (0/1 shards):   0%|          | 0/1144 [00:00<?, ? examples/s]

  ✅ test: 1,144 windows → /content/drive/MyDrive/yeastcaduceus/data/mlm_dataset/test

✅ Cell 2 complete — MLM dataset saved to /content/drive/MyDrive/yeastcaduceus/data/mlm_dataset


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2b — Patch: add repeat mask to MLM dataset
#
# Cell 2 uppercased all sequences before saving — soft-mask information lost.
# PlantCAD2 paper explicitly down-weights repetitive positions in MLM loss.
# This cell rebuilds the MLM dataset with:
#   - 'sequence'   : uppercase (unchanged, required by tokenizer)
#   - 'is_repeat'  : bool list, True where original sequence was lowercase
#
# Phase 2 training loop uses is_repeat to zero/down-weight loss at repeat positions:
#   loss_weight[is_repeat] *= 0.1
#
# Runtime: ~15-20 min on Colab (same 165 FASTAs, same windows — no recomputation
# of window boundaries, just one extra field per record).
# Overwrites /content/drive/MyDrive/yeastcaduceus/data/mlm_dataset in-place.
# ─────────────────────────────────────────────────────────────────────────────

from pathlib import Path
from datasets import Dataset, load_from_disk
from tqdm import tqdm

MLM_DATASET_PATH = Path('/content/drive/MyDrive/yeastcaduceus/data/mlm_dataset')
FASTA_DIR_165    = Path('/content/drive/MyDrive/shorkie/data/backup/genomes/165_Saccharomycetales')

# Reuse constants from Cell 1
# WINDOW_SIZE, STRIDE, REPEAT_THRESH, VAL_CHROMS, TEST_CHROMS must be in scope.
# If kernel restarted, re-run Cell 1 first.

def parse_fasta_raw(fasta_path):
    """Parse FASTA preserving original case (soft-mask info intact)."""
    chroms = {}
    current_name = None
    current_seq = []
    with open(fasta_path) as f:
        for line in f:
            line = line.rstrip()
            if line.startswith('>'):
                if current_name:
                    chroms[current_name] = ''.join(current_seq)
                current_name = line[1:].split()[0]
                current_seq = []
            else:
                current_seq.append(line)
    if current_name:
        chroms[current_name] = ''.join(current_seq)
    return chroms

def windows_with_repeat_mask(chrom_name, raw_seq, split, genome_id):
    """
    Slide window over raw_seq (mixed case).
    Returns records with:
      - sequence   : uppercase string (tokenizer input)
      - is_repeat  : list of bool, True = softmasked position
    Filters windows where repeat fraction > REPEAT_THRESH.
    """
    records = []
    seq_len = len(raw_seq)
    for start in range(0, seq_len - WINDOW_SIZE + 1, STRIDE):
        window_raw = raw_seq[start:start + WINDOW_SIZE]
        is_repeat  = [c.islower() for c in window_raw]
        repeat_frac = sum(is_repeat) / WINDOW_SIZE
        if repeat_frac > REPEAT_THRESH:
            continue
        records.append({
            'genome_id':  genome_id,
            'chrom':      chrom_name,
            'start':      start,
            'end':        start + WINDOW_SIZE,
            'split':      split,
            'sequence':   window_raw.upper(),
            'is_repeat':  is_repeat,           # NEW: bool list, len=8192
        })
    return records

# ── Rebuild all splits ────────────────────────────────────────────────────────
fasta_files = sorted(FASTA_DIR_165.glob('*.softmask'))
print(f'Rebuilding MLM dataset with repeat mask ({len(fasta_files)} FASTAs)...')
print(f'Output: {MLM_DATASET_PATH}')

split_records = {'train': [], 'val': [], 'test': []}

for fasta_path in tqdm(fasta_files, desc='FASTAs'):
    genome_id = fasta_path.name.split('.')[0]
    raw_chroms = parse_fasta_raw(fasta_path)

    for chrom_name, raw_seq in raw_chroms.items():
        # Val/test only from R64 — skip non-R64 chroms for val/test assignment
        split = chrom_to_split(chrom_name, is_r64=False)
        records = windows_with_repeat_mask(chrom_name, raw_seq, split, genome_id)
        for r in records:
            split_records[r['split']].append(r)

print('\nWindow counts:')
for split, records in split_records.items():
    print(f'  {split:5s}: {len(records):,}')

# ── Save (overwrites existing MLM dataset) ────────────────────────────────────
print('\nSaving updated MLM dataset...')
for split, records in split_records.items():
    if not records:
        continue
    ds = Dataset.from_list(records)
    out_path = MLM_DATASET_PATH / split
    ds.save_to_disk(str(out_path))
    print(f'  ✅ {split}: {len(ds):,} → {out_path}')
    print(f'     Fields: {list(ds.features.keys())}')

# ── Sanity check ──────────────────────────────────────────────────────────────
print('\nSanity check — sample window:')
ds_train = load_from_disk(str(MLM_DATASET_PATH / 'train'))
sample = ds_train[0]
is_rep = sample['is_repeat']
rep_frac = sum(is_rep) / len(is_rep)
print(f'  sequence[:20]  : {sample["sequence"][:20]}')
print(f'  is_repeat[:20] : {is_rep[:20]}')
print(f'  repeat fraction: {rep_frac:.3f} (should be < {REPEAT_THRESH})')
assert len(is_rep) == WINDOW_SIZE, f'Expected {WINDOW_SIZE} bools, got {len(is_rep)}'
assert all(c == c.upper() for c in sample['sequence'] if c.isalpha()), \
    'sequence field contains lowercase — tokenizer will fail'

print(f'\n✅ Cell 2b complete — MLM dataset rebuilt with is_repeat field')
print(f'   Phase 2 loss: weight 1.0 for non-repeat, 0.1 for repeat positions')

Rebuilding MLM dataset with repeat mask (165 FASTAs)...
Output: /content/drive/MyDrive/yeastcaduceus/data/mlm_dataset


FASTAs: 100%|██████████| 165/165 [07:38<00:00,  2.78s/it]



Window counts:
  train: 848,929
  val  : 0
  test : 0

Saving updated MLM dataset...


Saving the dataset (0/16 shards):   0%|          | 0/848929 [00:00<?, ? examples/s]

  ✅ train: 848,929 → /content/drive/MyDrive/yeastcaduceus/data/mlm_dataset/train
     Fields: ['genome_id', 'chrom', 'start', 'end', 'split', 'sequence', 'is_repeat']

Sanity check — sample window:
  sequence[:20]  : AACCGGCGCCAGTGTGCTGG
  is_repeat[:20] : [False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False]
  repeat fraction: 0.053 (should be < 0.07)

✅ Cell 2b complete — MLM dataset rebuilt with is_repeat field
   Phase 2 loss: weight 1.0 for non-repeat, 0.1 for repeat positions


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2b — Patch: rebuild MLM dataset with is_repeat field
#
# Cell 2 uppercased all sequences before saving — soft-mask info lost.
# PlantCAD2 paper explicitly down-weights repetitive positions in MLM loss.
# This cell rebuilds the MLM dataset with:
#   - 'sequence'  : uppercase (unchanged, required by tokenizer)
#   - 'is_repeat' : list[bool], len=8192, True where original was lowercase
#
# Phase 2 training loop: loss_weight = 1.0 - 0.9 * is_repeat
#
# Runtime: ~8 min on Colab A100.
# Overwrites /content/drive/MyDrive/yeastcaduceus/data/mlm_dataset in-place.
#
# Prerequisites: Cell 1 must be in scope (WINDOW_SIZE, STRIDE, REPEAT_THRESH,
#                VAL_CHROMS, TEST_CHROMS, chrom_to_split).
# ─────────────────────────────────────────────────────────────────────────────

from pathlib import Path
from datasets import Dataset, load_from_disk
from tqdm import tqdm

MLM_DATASET_PATH = Path('/content/drive/MyDrive/yeastcaduceus/data/mlm_dataset')
FASTA_DIR_165    = Path('/content/drive/MyDrive/shorkie/data/backup/genomes/165_Saccharomycetales')
R64_ACCESSION    = 'GCA_000146045'   # R64 genome prefix — used to set is_r64 flag

def parse_fasta_raw(fasta_path):
    """Parse FASTA preserving original case (soft-mask info intact)."""
    chroms = {}
    current_name = None
    current_seq = []
    with open(fasta_path) as f:
        for line in f:
            line = line.rstrip()
            if line.startswith('>'):
                if current_name:
                    chroms[current_name] = ''.join(current_seq)
                current_name = line[1:].split()[0]
                current_seq = []
            else:
                current_seq.append(line)
    if current_name:
        chroms[current_name] = ''.join(current_seq)
    return chroms

def windows_with_repeat_mask(chrom_name, raw_seq, split, genome_id):
    """
    Slide WINDOW_SIZE windows over raw_seq (mixed case).
    Applies same repeat filter as Cell 2 (>7% lowercase → drop).
    Returns records with uppercase sequence + is_repeat bool list.
    """
    records = []
    for start in range(0, len(raw_seq) - WINDOW_SIZE + 1, STRIDE):
        window_raw  = raw_seq[start:start + WINDOW_SIZE]
        is_repeat   = [c.islower() for c in window_raw]
        repeat_frac = sum(is_repeat) / WINDOW_SIZE
        if repeat_frac > REPEAT_THRESH:
            continue
        # Drop N-dominated windows (>50%) — same filter as Cell 2
        n_frac = window_raw.upper().count('N') / WINDOW_SIZE
        if n_frac > 0.5:
            continue
        records.append({
            'genome_id': genome_id,
            'chrom':     chrom_name,
            'start':     start,
            'end':       start + WINDOW_SIZE,
            'split':     split,
            'sequence':  window_raw.upper(),
            'is_repeat': is_repeat,
        })
    return records

# ── Main loop ─────────────────────────────────────────────────────────────────
fasta_files = sorted(FASTA_DIR_165.glob('*.softmask'))
print(f'Rebuilding MLM dataset with is_repeat field ({len(fasta_files)} FASTAs)...')
print(f'Output: {MLM_DATASET_PATH}')

split_records = {'train': [], 'val': [], 'test': []}

for fasta_path in tqdm(fasta_files, desc='FASTAs'):
    genome_id = fasta_path.name.split('.')[0]
    is_r64    = R64_ACCESSION in fasta_path.name   # ← fix: was hardcoded False

    raw_chroms = parse_fasta_raw(fasta_path)
    for chrom_name, raw_seq in raw_chroms.items():
        split   = chrom_to_split(chrom_name, is_r64=is_r64)
        records = windows_with_repeat_mask(chrom_name, raw_seq, split, genome_id)
        for r in records:
            split_records[r['split']].append(r)

print('\nWindow counts:')
for split, records in split_records.items():
    print(f'  {split:5s}: {len(records):,}')

# Sanity — must match Cell 2 counts exactly
assert split_records['train'], 'train is empty — something went wrong'
assert 1100 <= len(split_records['val'])  <= 1200, \
    f'val count unexpected: {len(split_records["val"])} (expected ~1,121)'
assert 1100 <= len(split_records['test']) <= 1200, \
    f'test count unexpected: {len(split_records["test"])} (expected ~1,144)'

# ── Save (overwrites existing MLM dataset) ────────────────────────────────────
print('\nSaving updated MLM dataset...')
for split, records in split_records.items():
    if not records:
        print(f'  ⚠️  {split}: 0 records — skipping')
        continue
    ds = Dataset.from_list(records)
    out_path = MLM_DATASET_PATH / split
    ds.save_to_disk(str(out_path))
    print(f'  ✅ {split}: {len(ds):,} → {out_path}')
    print(f'     Fields: {list(ds.features.keys())}')

# ── Sanity check: load back and verify is_repeat ──────────────────────────────
print('\nSanity check — sample window from train:')
ds_train = load_from_disk(str(MLM_DATASET_PATH / 'train'))
sample   = ds_train[0]
is_rep   = sample['is_repeat']

print(f'  Fields present : {list(sample.keys())}')
print(f'  sequence[:20]  : {sample["sequence"][:20]}')
print(f'  is_repeat[:20] : {is_rep[:20]}')
print(f'  repeat fraction: {sum(is_rep)/len(is_rep):.4f} (must be ≤ {REPEAT_THRESH})')

assert 'is_repeat' in sample,              'is_repeat field missing'
assert len(is_rep) == WINDOW_SIZE,         f'is_repeat length {len(is_rep)} ≠ {WINDOW_SIZE}'
assert all(c == c.upper() for c in sample['sequence'] if c.isalpha()), \
    'sequence contains lowercase — tokenizer will fail'

print(f'\n✅ Cell 2b complete')
print(f'   MLM dataset rebuilt: {len(ds_train):,} train windows with is_repeat field')
print(f'   Phase 2 loss: weight 1.0 for non-repeat, 0.1 for repeat positions')

Rebuilding MLM dataset with is_repeat field (165 FASTAs)...
Output: /content/drive/MyDrive/yeastcaduceus/data/mlm_dataset


FASTAs: 100%|██████████| 165/165 [09:35<00:00,  3.49s/it]



Window counts:
  train: 846,664
  val  : 1,121
  test : 1,144

Saving updated MLM dataset...


Saving the dataset (0/16 shards):   0%|          | 0/846664 [00:00<?, ? examples/s]

  ✅ train: 846,664 → /content/drive/MyDrive/yeastcaduceus/data/mlm_dataset/train
     Fields: ['genome_id', 'chrom', 'start', 'end', 'split', 'sequence', 'is_repeat']


Saving the dataset (0/1 shards):   0%|          | 0/1121 [00:00<?, ? examples/s]

  ✅ val: 1,121 → /content/drive/MyDrive/yeastcaduceus/data/mlm_dataset/val
     Fields: ['genome_id', 'chrom', 'start', 'end', 'split', 'sequence', 'is_repeat']


Saving the dataset (0/1 shards):   0%|          | 0/1144 [00:00<?, ? examples/s]

  ✅ test: 1,144 → /content/drive/MyDrive/yeastcaduceus/data/mlm_dataset/test
     Fields: ['genome_id', 'chrom', 'start', 'end', 'split', 'sequence', 'is_repeat']

Sanity check — sample window from train:
  Fields present : ['genome_id', 'chrom', 'start', 'end', 'split', 'sequence', 'is_repeat']
  sequence[:20]  : AACCGGCGCCAGTGTGCTGG
  is_repeat[:20] : [False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False]
  repeat fraction: 0.0533 (must be ≤ 0.07)

✅ Cell 2b complete
   MLM dataset rebuilt: 846,664 train windows with is_repeat field
   Phase 2 loss: weight 1.0 for non-repeat, 0.1 for repeat positions


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — Supervised Dataset: R64 sequences + BigWig coverage targets
#
# For each 8192bp window in R64 (same windows as MLM dataset above):
#   - Extract sequence from R64 FASTA
#   - For each BigWig track: extract coverage at 16bp bins → 512-element vector
#   - Coverage is log1p-normalised per track (Shorkie convention)
#   - Store as .npz per genomic window — (n_tracks, 512) arrays
#
# NOTE: Full BigWig extraction for 5,249 tracks × all R64 windows is slow.
# This cell builds the SEQUENCE dataset and creates the BigWig MANIFEST.
# Actual coverage extraction runs in Phase 3 immediately before training
# to avoid storing 100GB+ of pre-extracted arrays on Drive.
#
# Phase 3 will stream BigWig coverage on-the-fly during DataLoader iteration.
# ─────────────────────────────────────────────────────────────────────────────

import pyBigWig
from datasets import Dataset

# ── Step 3a: R64 sequence windows ─────────────────────────────────────────────
print('Building R64 sequence windows for supervised training...')
assert R64_FASTA is not None, 'R64 FASTA not found — check FASTA_R64 path'

r64_chroms = parse_fasta(R64_FASTA)  # reuse parse_fasta from Cell 2
print(f'  R64 chromosomes: {sorted(r64_chroms.keys())}')

r64_records = []
for chrom_name, seq in sorted(r64_chroms.items()):
    split = chrom_to_split(chrom_name, is_r64=True)
    records = windows_from_chrom(chrom_name, seq, split, genome_id='R64')
    r64_records.extend(records)

print(f'  Total R64 windows: {len(r64_records):,}')
split_counts = {s: sum(1 for r in r64_records if r['split'] == s)
                for s in ['train', 'val', 'test']}
for s, n in split_counts.items():
    print(f'    {s}: {n:,}')

# Save R64 sequence dataset
for split in ['train', 'val', 'test']:
    split_records = [r for r in r64_records if r['split'] == split]
    if not split_records:
        continue
    ds = Dataset.from_list(split_records)
    out_path = OUT_SUP_SEQ / split
    ds.save_to_disk(str(out_path))
    print(f'  ✅ Supervised sequences {split}: {len(ds):,} → {out_path}')

# ── Step 3b: BigWig manifest ───────────────────────────────────────────────────
print('\nBuilding BigWig manifest...')
bw_files = sorted(BIGWIG_DIR.glob('*.bw'))
print(f'  Found {len(bw_files):,} BigWig files')

# Spot-check first 5: can they be opened? What chromosomes do they contain?
print('  Spot-checking first 5 BigWig files...')
bw_chrom_sets = []
valid_bw = []
corrupt_bw = []

for bw_path in bw_files[:5]:
    try:
        bw = pyBigWig.open(str(bw_path))
        chroms_in_bw = list(bw.chroms().keys())
        bw.close()
        bw_chrom_sets.append(set(chroms_in_bw))
        valid_bw.append(bw_path.name)
        print(f'    ✅ {bw_path.name}: {len(chroms_in_bw)} chroms, e.g. {chroms_in_bw[:3]}')
    except Exception as e:
        corrupt_bw.append(bw_path.name)
        print(f'    ❌ {bw_path.name}: {e}')

# Check chrom name compatibility with R64 FASTA
if bw_chrom_sets:
    bw_chroms_example = bw_chrom_sets[0]
    r64_chrom_names = set(r64_chroms.keys())
    overlap = r64_chrom_names & bw_chroms_example
    print(f'\n  R64 chrom names  : {sorted(r64_chrom_names)[:5]}...')
    print(f'  BigWig chrom names: {sorted(bw_chroms_example)[:5]}...')
    print(f'  Name overlap     : {len(overlap)} chroms match')
    if len(overlap) == 0:
        print('  ⚠️  ZERO OVERLAP — BigWig and FASTA chrom names differ!')
        print('  Will need chrom name remapping in Phase 3.')
        # Infer mapping: BigWig likely uses 'chrI', FASTA may use 'I' or 'chr1'
        print(f'  R64 example   : {list(r64_chrom_names)[:3]}')
        print(f'  BigWig example: {sorted(bw_chroms_example)[:3]}')

# Build full manifest CSV
bw_manifest = []
print('\nScanning all BigWig files for manifest (may take 1-2 min)...')
for bw_path in tqdm(bw_files, desc='BigWigs'):
    try:
        bw = pyBigWig.open(str(bw_path))
        n_chroms = len(bw.chroms())
        bw.close()
        bw_manifest.append({'filename': bw_path.name, 'status': 'ok',
                            'n_chroms': n_chroms})
    except Exception as e:
        bw_manifest.append({'filename': bw_path.name, 'status': f'error: {e}',
                            'n_chroms': 0})

bw_df = pd.DataFrame(bw_manifest)
n_ok     = (bw_df['status'] == 'ok').sum()
n_err    = (bw_df['status'] != 'ok').sum()
bw_df.to_csv(OUT_MANIFEST / 'bigwig_manifest.csv', index=False)

print(f'\n  ✅ BigWig manifest saved: {n_ok:,} OK, {n_err} errors')
if n_err > 0:
    print('  Corrupt/missing files:')
    print(bw_df[bw_df['status'] != 'ok'][['filename', 'status']].to_string())

print(f'\n✅ Cell 3 complete')

Building R64 sequence windows for supervised training...
  R64 chromosomes: ['chrI', 'chrII', 'chrIII', 'chrIV', 'chrIX', 'chrV', 'chrVI', 'chrVII', 'chrVIII', 'chrX', 'chrXI', 'chrXII', 'chrXIII', 'chrXIV', 'chrXV', 'chrXVI']
  Total R64 windows: 4,867
    train: 2,602
    val: 1,121
    test: 1,144


Saving the dataset (0/1 shards):   0%|          | 0/2602 [00:00<?, ? examples/s]

  ✅ Supervised sequences train: 2,602 → /content/drive/MyDrive/yeastcaduceus/data/supervised_seqs/train


Saving the dataset (0/1 shards):   0%|          | 0/1121 [00:00<?, ? examples/s]

  ✅ Supervised sequences val: 1,121 → /content/drive/MyDrive/yeastcaduceus/data/supervised_seqs/val


Saving the dataset (0/1 shards):   0%|          | 0/1144 [00:00<?, ? examples/s]

  ✅ Supervised sequences test: 1,144 → /content/drive/MyDrive/yeastcaduceus/data/supervised_seqs/test

Building BigWig manifest...
  Found 4,201 BigWig files
  Spot-checking first 5 BigWig files...
    ✅ AAP1_S0_coverage.bw: 16 chroms, e.g. ['chrI', 'chrII', 'chrIII']
    ✅ AAP1_S1_coverage.bw: 16 chroms, e.g. ['chrI', 'chrII', 'chrIII']
    ✅ ABD1_S0_coverage.bw: 16 chroms, e.g. ['chrI', 'chrII', 'chrIII']
    ✅ ABF1_S0_coverage.bw: 16 chroms, e.g. ['chrI', 'chrII', 'chrIII']
    ✅ ABF1_S1_coverage.bw: 16 chroms, e.g. ['chrI', 'chrII', 'chrIII']

  R64 chrom names  : ['chrI', 'chrII', 'chrIII', 'chrIV', 'chrIX']...
  BigWig chrom names: ['chrI', 'chrII', 'chrIII', 'chrIV', 'chrIX']...
  Name overlap     : 16 chroms match

Scanning all BigWig files for manifest (may take 1-2 min)...


BigWigs:  63%|██████▎   | 2643/4201 [1:28:48<43:10,  1.66s/it]

In [ ]:
import json
import pandas as pd
from pathlib import Path

# Paths — adjust if yours differ
manifest_csv = Path('/content/drive/MyDrive/yeastcaduceus/data/manifests/bigwig_manifest.csv')
bw_dir       = Path('/content/drive/MyDrive/shorkie/data/backup/supervised/bigwigs')

df = pd.read_csv(manifest_csv)

rows = []
for i, row in df.iterrows():
    name  = Path(row['filename']).stem   # e.g. AAP1_S0_coverage
    parts = name.split('_')
    rows.append({
        'track_index': i,
        'filename':    row['filename'],
        'gene':        parts[0] if len(parts) >= 1 else name,
        'condition':   parts[1] if len(parts) >= 2 else 'unknown',
    })

manifest = {
    'bw_dir':                   str(bw_dir),
    'n_tracks':                 len(rows),
    'bin_size':                 16,
    'bins_per_window':          512,
    'chrom_overlap_confirmed':  16,
    'tracks':                   rows,
}

out = manifest_csv.parent / 'bigwig_manifest.json'
with open(out, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f'✅ {len(rows):,} tracks → {out}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — Statistics + Go / No-Go
#
# Validates dataset sizes and chrom name compatibility before Phase 2.
# Also prints a tokenizer smoke-check: tokenize 3 random MLM windows
# to confirm the HF Dataset integrates cleanly with CAD2-Small's tokenizer.
# ─────────────────────────────────────────────────────────────────────────────

from datasets import load_from_disk
from transformers import AutoTokenizer
import random

MODEL_ID = 'kuleshov-group/PlantCAD2-Small-l24-d0768'

print('='*60)
print('PHASE 1 SUMMARY')
print('='*60)

# ── MLM dataset sizes ─────────────────────────────────────────────────────────
print('\nMLM Dataset (165_Saccharomycetales):')
mlm_sizes = {}
for split in ['train', 'val', 'test']:
    path = OUT_MLM / split
    if path.exists():
        ds = load_from_disk(str(path))
        mlm_sizes[split] = len(ds)
        print(f'  {split:5s}: {len(ds):>10,} windows')
    else:
        mlm_sizes[split] = 0
        print(f'  {split:5s}: ❌ not found')

# ── Supervised dataset sizes ──────────────────────────────────────────────────
print('\nSupervised Dataset (R64):')
sup_sizes = {}
for split in ['train', 'val', 'test']:
    path = OUT_SUP_SEQ / split
    if path.exists():
        ds = load_from_disk(str(path))
        sup_sizes[split] = len(ds)
        print(f'  {split:5s}: {len(ds):>10,} windows')
    else:
        sup_sizes[split] = 0
        print(f'  {split:5s}: ❌ not found')

# ── BigWig manifest summary ───────────────────────────────────────────────────
print('\nBigWig Manifest:')
bw_manifest_path = OUT_MANIFEST / 'bigwig_manifest.csv'
if bw_manifest_path.exists():
    bw_df = pd.read_csv(bw_manifest_path)
    n_ok  = (bw_df['status'] == 'ok').sum()
    n_err = (bw_df['status'] != 'ok').sum()
    print(f'  Total tracks: {len(bw_df):,}')
    print(f'  OK:           {n_ok:,}')
    print(f'  Corrupt:      {n_err}')
    print(f'  (Shorkie used 5,215 — we have {len(bw_df):,})')

# ── Tokenizer smoke-check ─────────────────────────────────────────────────────
print('\nTokenizer smoke-check (3 random MLM windows):')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
train_ds = load_from_disk(str(OUT_MLM / 'train'))
indices = random.sample(range(len(train_ds)), 3)

for idx in indices:
    seq = train_ds[idx]['sequence']
    tokens = tokenizer(seq, return_tensors='pt')['input_ids']
    assert tokens.shape == (1, WINDOW_SIZE), f'Unexpected token shape: {tokens.shape}'
    print(f'  idx {idx:>7}: genome={train_ds[idx]["genome_id"]:20s}  '
          f'chrom={train_ds[idx]["chrom"]:8s}  '
          f'tokens={tokens.shape}  ✅')

# ── Go / No-Go ────────────────────────────────────────────────────────────────
print(f'\n{"="*60}')
print('GO / NO-GO')
checks = {
    'MLM train windows > 100k'   : mlm_sizes.get('train', 0) > 100_000,
    'MLM val windows > 1k'       : mlm_sizes.get('val', 0) > 1_000,
    'MLM test windows > 1k'      : mlm_sizes.get('test', 0) > 1_000,
    'Supervised train > 1k'      : sup_sizes.get('train', 0) > 1_000,
    'BigWig manifest exists'     : bw_manifest_path.exists(),
    'Tokenizer shape correct'    : True,  # would have asserted above
}
all_pass = True
for check, result in checks.items():
    print(f'  {"✅" if result else "❌"} {check}')
    if not result:
        all_pass = False

print()
if all_pass:
    print('✅ Phase 1 PASSED — ready for Phase 2 (MLM domain adaptation)')
else:
    print('❌ Phase 1 FAILED — address failed checks above before proceeding')